In [411]:
import pandas as pd 
import numpy as np 
import re
from tableone import TableOne

pd.set_option('display.max_rows', 100)  # Show up to 100 rows
pd.set_option('display.max_columns', None)

In [412]:
path = '/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/Data extract/KMCPPD2024_DATA_LABELS_2026-10-07_1128.csv'
df = pd.read_csv(path)

/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/1923387543.py:2: DtypeWarning: Columns (8,25,26,27,28,29,30,31,32,33,34,75,79,80,85,86,87,88,89,90,92,93,94,95,96,97,98,110,111,112,113,115,116,117,118,119,121,122,124,126,127,128,129,130,131,132,133,134,135,136,137,144,145,146,147,152,155,156,158,160,162,164,166,168,169,172,173,176,179,180,181,182,183,184,185,186,187,188,189,190,191,192,194,199,200,203,204,206,209,210,212,214,216,218,220,222,223,224,225,226,227,239,243,245,279,280,283,284,285,289,290,291,294,295,303,304,306,308,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,335,336,337,338,339,340,341,342,343,344,345,347,349,350,351,353,354,355,356,357,359,372,376,378,410,412,416,417,418,422,423,424,427,429,430,439,440,442,444,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,472,473,474,475,476,477,478,479,480,481,482,484,486,487,488,489,491,492,493,494,496,498,499,501,502,503,504

In [413]:
## Combine participant crfs into one record

def _combine_series(s):
    vals = s.dropna().astype(str).unique()
    if vals.size == 0:
        return np.nan
    if vals.size == 1:
        v = vals[0]
        if pd.api.types.is_numeric_dtype(s):
            try:
                return pd.to_numeric(v)
            except:
                return v
        if pd.api.types.is_bool_dtype(s):
            return v.lower() in ("true", "1", "yes")
        return v
    return " | ".join(vals)

dedup_df = df.groupby("participant_id", sort=False, as_index=False).agg(_combine_series)
print(f"rows before: {len(df)}, rows after: {len(dedup_df)}")
df = dedup_df.copy()

rows before: 7736, rows after: 612


/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/2991340239.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dedup_df = df.groupby("participant_id", sort=False, as_index=False).agg(_combine_series)


In [414]:
df = df.copy()  # Create a copy of the DataFrame to avoid SettingWithCopyWarning

In [415]:
df['Cohort'] = df['redcap_event_name'].map({'Control (NICU → KMC ward for < 48 hours → home)': 'Control', 'Intervention (Immediate admission to the KMC Ward → home)': 'Intervention'})

In [416]:
df['redcap_event_name_ed'] = df['redcap_event_name'].replace(r"\b(?!control\b)\w+", '', regex=True)
df['redcap_event_name_ed'].value_counts()

redcap_event_name_ed
( : ) |  ( : ) |        ( : ) |          ( : ) |        ( : ) |        ( : ) |  ( : )                   244
( : ) |  ( : ) |        ( : ) |          ( : ) |        ( : ) |  ( : )                                  228
( : ) |  ( : ) |        ( : ) |          ( : ) |  ( : )                                                  70
( : ) |  ( : ) |        ( : ) |          ( : ) |        ( : ) |        ( : ) |        ( : ) |  ( : )     33
( : ) |  ( : ) |        ( : ) |  ( : )                                                                   18
( : ) |  ( : ) |  ( : )                                                                                   9
( : ) |  ( : ) |        ( : ) |        ( : ) |  ( : )                                                     7
( : ) |  ( : ) |        ( : ) |        ( : ) |        ( : ) |  ( : )                                      2
( : ) |  ( : ) |          ( : ) |        ( : ) |  ( : )                                                   1
Name: c

In [417]:
# Convert date strings to datetime
df["en_date"] = pd.to_datetime(df["en_date"], errors="coerce")
df["en_m_dob"] = pd.to_datetime(df["en_m_dob"], errors="coerce")

# Calculate age at enrollment in completed years (no decimal places)
df["mothers_age"] = (
    np.floor((df["en_date"] - df["en_m_dob"]).dt.days / 365.25)
    .astype("Int64")
)
df['mothers_age_group'] = pd.cut(df['mothers_age'], 
                                  bins=[14, 19, 24, 29, 34, 39, 100],
                                  labels=['15-19 years', '20-24 years', '25-29 years', 
                                          '30-34 years', '35-39 years', '40+ years'])

/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/3991350160.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["en_date"] = pd.to_datetime(df["en_date"], errors="coerce")
/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/3991350160.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["en_m_dob"] = pd.to_datetime(df["en_m_dob"], errors="coerce")


In [418]:
# work on parity and gravida
df["en_para"] = pd.to_numeric(df["en_para"], errors="coerce")
df['mothers_par_group'] = pd.cut(df['en_para'], 
                                  bins=[-1,0,1,2, 100],
                                  labels=['0','1', '2', '3+'])

# Work on gravida
df["en_gra"] = pd.to_numeric(df["en_gra"], errors="coerce")
df['mothers_grav_group'] = pd.cut(df['en_gra'], 
                                  bins=[-1,0,1,2, 100],
                                  labels=['0','1', '2', '3+'])          


# Sexual debut age categorization
df["en_m_asxd"] = pd.to_numeric(df["en_m_asxd"], errors="coerce")
df['mothers_sex_debut_group'] = pd.cut(df['en_m_asxd'],
                                       bins=[-1, 14, 18, 100],
                                       labels=['<15 years', '15-18 years', ">18 years"])
# Number of ANC visits categorization
df["en_m_anc"] = pd.to_numeric(df["en_m_anc"], errors="coerce")
df['mothers_anc_group'] = pd.cut(df['en_m_anc'],
                                  bins=[-1, 2, 4, 6, 100],
                                  labels=['0-2 Vists','3-4 Vists','5-6 Vists','7+ Vists'])
# EPDS score categorization
df["fp_m_scr_epds_sco"] = pd.to_numeric(df["fp_m_scr_epds_sco"], errors="coerce")
df['mothers_epds_group_2weeks'] = pd.cut(df['fp_m_scr_epds_sco'],
                                  bins=[-1,10, 100],
                                  labels=[ 'Negative', 'Positive'])

# EPDS score categorization 6-8 weeks
df["fp1_m_scr_epds_sco"] = pd.to_numeric(df["fp1_m_scr_epds_sco"], errors="coerce")
df['mothers_epds_group_6-8weeks'] = pd.cut(df['fp1_m_scr_epds_sco'],
                                  bins=[-1,10, 100],
                                  labels=[ 'Negative', 'Positive'])    


In [419]:
# mode of delivery
df["en_mod"] = df["en_mod"].replace({'Assisted Delivery (forceps or vacuum)': np.nan})
# family history of physilogical disorder
df["en_m_psyc"] = df["en_m_psyc"].replace({"Don't Know": np.nan}, regex=True)
df["en_mod"].unique()

array(['Spontaneous Vaginal Delivery', 'Cesarean Section', nan],
      dtype=object)

In [420]:

# ============================================================
# SELF-HARM VARIABLE RECODING
# ============================================================

# 2 weeks
df["fp_m_epds_harming_self"] = df["fp_m_epds_harming_self"].replace({
    "Hardly ever": "Positive(hardly-ever/Never/Sometimes)",
    "Yes, quite often": "Positive(hardly-ever/Never/Sometimes)",
    "Sometimes": "Positive(hardly-ever/Never/Sometimes)",
    "Never": "Negative"
})

# 6-8 weeks
df["fp1_m_epds_harming_self"] = df["fp1_m_epds_harming_self"].replace({
    "Hardly ever": "Positive(hardly-ever/Never/Sometimes)",
    "Yes, quite often": "Positive(hardly-ever/Never/Sometimes)",
    "Sometimes": "Positive(hardly-ever/Never/Sometimes)",
    "Never": "Negative"
})



# ============================================================
# ANXIETY SCORE - 2 WEEKS
# ============================================================

# Question 3:
# I have blamed myself unnecessarily when things went wrong
df["fp_m_epds_blamed_self_num"] = df["fp_m_epds_blamed_self"].map({
    "Yes, most of the time": 3,
    "Yes, some of the time": 2,
    "Not very often": 1,
    "No, never": 0
})


# Question 4:
# I have been anxious or worried for no good reason
df["fp_m_epds_anxious_num"] = df["fp_m_epds_anxious"].map({
    "No, not at all": 0,
    "Hardly ever": 1,
    "Yes, sometimes": 2,
    "Yes, very often": 3
})


# Question 5:
# I have felt scared or panicky for no very good reason
df["fp_m_epds_scared_num"] = df["fp_m_epds_scared"].map({
    "Yes, quite a lot": 3,
    "Yes, sometimes": 2,
    "No, not much": 1,
    "No, not at all": 0
})


# ============================================================
# CREATE 2-WEEK ANXIETY SCORE
# ============================================================

df["anxiety_score_2weeks"] = (
    df["fp_m_epds_blamed_self_num"]
    + df["fp_m_epds_anxious_num"]
    + df["fp_m_epds_scared_num"]
)


# Create Positive / Negative result
df["anxiety_result_2weeks"] = np.select(
    [
        df["anxiety_score_2weeks"] > 4,
        df["anxiety_score_2weeks"] <= 4
    ],
    [
        "Positive",
        "Negative"
    ],
    default=None
)

# ============================================================
# ANXIETY SCORE - 6-8 WEEKS
# ============================================================

# Question 3:
# I have blamed myself unnecessarily when things went wrong
df["fp1_m_epds_blamed_self_num"] = df["fp1_m_epds_blamed_self"].map({
    "Yes, most of the time": 3,
    "Yes, some of the time": 2,
    "Not very often": 1,
    "No, never": 0
})


# Question 4:
# I have been anxious or worried for no good reason
df["fp1_m_epds_anxious_num"] = df["fp1_m_epds_anxious"].map({
    "No, not at all": 0,
    "Hardly ever": 1,
    "Yes, sometimes": 2,
    "Yes, very often": 3
})


# Question 5:
# I have felt scared or panicky for no very good reason
df["fp1_m_epds_scared_num"] = df["fp1_m_epds_scared"].map({
    "Yes, quite a lot": 3,
    "Yes, sometimes": 2,
    "No, not much": 1,
    "No, not at all": 0
})


# ============================================================
# CREATE 6-8 WEEK ANXIETY SCORE
# ============================================================

df["anxiety_score_6to8weeks"] = (
    df["fp1_m_epds_blamed_self_num"]
    + df["fp1_m_epds_anxious_num"]
    + df["fp1_m_epds_scared_num"]
)


# Create Positive / Negative result
df["anxiety_result_6to8weeks"] = np.select(
    [
        df["anxiety_score_6to8weeks"] > 4,
        df["anxiety_score_6to8weeks"] <= 4
    ],
    [
        "Positive",
        "Negative"
    ],
    default=None
)


In [421]:
df["en_rand"] = df["en_rand"].replace({
    "Control (NICU → KMC ward for < 48 hours → home)": "Control",
    "Intervention (Immediate admission to the KMC Ward → home)": "Intervention"
})

In [422]:
## Save cleaned dataset
df.to_csv("/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/Data extract/Latest Extract/Clean_dataset.csv", index=False)


In [423]:
columns = [ 
    "mothers_age",
    "mothers_age_group",
    "en_m_mstat",
    "en_m_sku",
    "en_m_inc",
    "en_m_alc",
    "en_m_psyc",
    "en_m_hiv",
    "mothers_anc_group",
    "en_mod",
    "mothers_grav_group",
    "mothers_epds_group_2weeks",
    'mothers_epds_group_6-8weeks',
    'fp_m_epds_harming_self',
    'fp1_m_epds_harming_self',
    'anxiety_result_2weeks',
    'anxiety_result_6to8weeks'
]
categorical = [
    "mothers_age_group",
    "en_m_mstat",
    "en_m_sku",
    "en_m_inc",
    "en_m_alc",
    "en_m_psyc",
    "en_m_hiv",
    "mothers_anc_group",
    "en_mod",
    "mothers_grav_group",
    "mothers_epds_group_2weeks",
    'mothers_epds_group_6-8weeks',
    'fp_m_epds_harming_self',
    'fp1_m_epds_harming_self',
    'anxiety_result_2weeks',
    'anxiety_result_6to8weeks'
]

nonnormal = ["mothers_grav_group"]

min_max = ["mothers_age"]

continuous = [
    "mothers_age"
    ]


lables = {
    "mothers_age": "Mother's Age",
    "mothers_age_group": "Mother's Age Group",
    "mothers_grav_group": "Mother's Gravida Group",
    "en_mod": "Mode of Delivery",
    "en_m_mstat": "Mother's Marital Status",
    "en_m_hiv": "Mother's HIV Status",
    "en_m_sku": "Mother's Education Status",
    "en_m_inc": "Mother's Source of Income ",
    "mothers_anc_group": "Mother's ANC Visits",
    "en_m_alc": "Mother's Alcohol Use Status",
    "mothers_epds_group_2weeks": "Mother's EPDS Score (2 weeks)",
    "mothers_epds_group_6-8weeks": "Mother's EPDS Score (6-8 weeks)",
    "fp_m_epds_harming_self": "Mother's Self-Harm Score (2 weeks)",
    "fp1_m_epds_harming_self": "Mother's Self-Harm Score (6-8 weeks)",
    "anxiety_result_2weeks": "Mother's Anxiety Result (2 weeks)",
    "anxiety_result_6to8weeks": "Mother's Anxiety Result (6-8 weeks)"
}   

groupby = 'en_rand'

In [424]:
df["en_m_psyc"].value_counts(dropna=False)

en_m_psyc
No     543
Yes     41
NaN     28
Name: count, dtype: int64

In [425]:
mytable = TableOne(df, columns, categorical, groupby = groupby,rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)

/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not found: {k}")


In [426]:
mytable

Grouped by en_rand                                             
                                                                                             Missing     Overall     Control Intervention P-Value
n                                                                                                            612         306          306        
Mother's Age, mean (SD)                                                                            0  26.9 (6.5)  26.5 (6.2)   27.3 (6.7)   0.120
Mother's Age Group, n (%)                   15-19 years                                                84 (13.7)   45 (14.7)    39 (12.7)   0.265
                                            20-24 years                                               162 (26.5)   80 (26.1)    82 (26.8)        
                                            25-29 years                                               160 (26.1)   84 (27.5)    76 (24.8)        
                                            30-34 years                                               124 (20.3)   66 (21.6)    58 (19.0)        
                                            35-39 years                                                 58 (9.5)    21 (6.9)    37 (12.1)        
                                            40+ years                                                   24 (3.9)    10 (3.3)     14 (4.6)        
Mother's Marital Status, n (%)              Divorced                                                     1 (0.2)     1 (0.3)      0 (0.0)   0.653
                                            In a relationship                                         119 (19.4)   60 (19.6)    59 (19.3)        
                                            Married                                                   470 (76.8)  232 (75.8)   238 (77.8)        
                                            Separated                                                   10 (1.6)     6 (2.0)      4 (1.3)        
                                            Single                                                      11 (1.8)     7 (2.3)      4 (1.3)        
                                            Widowed                                                      1 (0.2)     0 (0.0)      1 (0.3)        
Mother's Education Status, n (%)            Completed college or university                            74 (12.1)   35 (11.4)    39 (12.7)   0.807
                                            Completed primary school education                         91 (14.9)   50 (16.3)    41 (13.4)        
                                            Completed secondary school education                       99 (16.2)   48 (15.7)    51 (16.7)        
                                            No education                                                15 (2.5)     5 (1.6)     10 (3.3)        
                                            Some college or university                                  27 (4.4)    15 (4.9)     12 (3.9)        
                                            Some primary school education                               36 (5.9)    18 (5.9)     18 (5.9)        
                                            Some secondary school education                           253 (41.3)  125 (40.8)   128 (41.8)        
                                            Vocational Training                                         17 (2.8)    10 (3.3)      7 (2.3)        
Mother's Source of Income , n (%)           Formal Employment                                          91 (14.9)   43 (14.1)    48 (15.7)   0.370
                                            Informal Employment                                       156 (25.5)   72 (23.5)    84 (27.5)        
                                            Unemployed                                                365 (59.6)  191 (62.4)   174 (56.9)        
Mother's Alcohol Use Status, n (%)          No                                                        532 (86.9)  262 (85.6)   270 (88.2)   

# Columns for outcome variables

In [393]:
columns = [ 
    "mothers_age",
    "mothers_age_group",
    "en_m_mstat",
    "en_m_sku",
    "en_m_inc",
    "en_m_alc",
    "en_m_psyc",
    "en_m_hiv",
    "mothers_anc_group",
    "en_mod",
    "mothers_grav_group"
]
categorical = [
    "mothers_age_group",
    "en_m_mstat",
    "en_m_sku",
    "en_m_inc",
    "en_m_alc",
    "en_m_psyc",
    "en_m_hiv",
    "mothers_anc_group",
    "en_mod",
    "mothers_grav_group"
]

nonnormal = ["mothers_grav_group"]

min_max = ["mothers_age"]

continuous = [
    "mothers_age"
    ]


lables = {
    "mothers_age": "Mother's Age",
    "mothers_age_group": "Mother's Age Group",
    "mothers_grav_group": "Mother's Gravida Group",
    "en_mod": "Mode of Delivery",
    "en_m_mstat": "Mother's Marital Status",
    "en_m_hiv": "Mother's HIV Status",
    "en_m_sku": "Mother's Education Status",
    "en_m_inc": "Mother's Source of Income ",
    "mothers_anc_group": "Mother's ANC Visits",
    "en_m_alc": "Mother's Alcohol Use Status"
}   

groupby = 'mothers_epds_group_2weeks'

In [394]:
df['mothers_epds_group_2weeks'] = df['mothers_epds_group_2weeks'].replace("None", np.nan,regex=True)
df_epds_2weeks = df.dropna(subset=['mothers_epds_group_2weeks'])
df_epds_2weeks['mothers_epds_group_2weeks'].value_counts(dropna=False)
# TableOne for mothers_epds_group_2weeks
mytable_epds_2weeks = TableOne(df_epds_2weeks, columns, categorical, groupby = groupby,rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_epds_2weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not

Grouped by mothers_epds_group_2weeks                                            
                                                                                                     Missing     Overall    Negative    Positive P-Value
n                                                                                                                    586         502          84        
Mother's Age, mean (SD)                                                                                    0  26.9 (6.5)  26.8 (6.4)  27.7 (6.8)   0.226
Mother's Age Group, n (%)          15-19 years                                                                 79 (13.5)   69 (13.7)   10 (11.9)   0.420
                                   20-24 years                                                                154 (26.3)  136 (27.1)   18 (21.4)        
                                   25-29 years                                                                155 (26.5)  129 (25.7)   26 (31.0)        
                                   30-34 years                                                                120 (20.5)  102 (20.3)   18 (21.4)        
                                   35-39 years                                                                  55 (9.4)    49 (9.8)     6 (7.1)        
                                   40+ years                                                                    23 (3.9)    17 (3.4)     6 (7.1)        
Mother's Marital Status, n (%)     Divorced                                                                      1 (0.2)     1 (0.2)     0 (0.0)   0.763
                                   In a relationship                                                          118 (20.1)  101 (20.1)   17 (20.2)        
                                   Married                                                                    445 (75.9)  382 (76.1)   63 (75.0)        
                                   Separated                                                                    10 (1.7)     7 (1.4)     3 (3.6)        
                                   Single                                                                       11 (1.9)    10 (2.0)     1 (1.2)        
                                   Widowed                                                                       1 (0.2)     1 (0.2)     0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                             74 (12.6)   64 (12.7)   10 (11.9)   0.202
                                   Completed primary school education                                          83 (14.2)   69 (13.7)   14 (16.7)        
                                   Completed secondary school education                                        96 (16.4)   81 (16.1)   15 (17.9)        
                                   No education                                                                 14 (2.4)    10 (2.0)     4 (4.8)        
                                   Some college or university                                                   27 (4.6)    20 (4.0)     7 (8.3)        
                                   Some primary school education                                                33 (5.6)    27 (5.4)     6 (7.1)        
                                   Some secondary school education                                            243 (41.5)  218 (43.4)   25 (29.8)        
                                   Vocational Training                                                          16 (2.7)    13 (2.6)     3 (3.6)        
Mother's Source of Income , n (%)  Formal Employment                                                           90 (15.4)   82 (16.3)     8 (9.5)   0.269
                                   Informal Employment                                                        151 (25.8)  127 (25.3)   24 (28.6)        
                                   Unemployed                                                 

In [395]:
df["mothers_epds_group_6-8weeks"].value_counts(dropna=False)

mothers_epds_group_6-8weeks
Negative    516
None         63
Positive     33
Name: count, dtype: int64

In [396]:
df['mothers_epds_group_6-8weeks'] = df['mothers_epds_group_6-8weeks'].replace("None", np.nan,regex=True)
df_epds_6to8weeks = df.dropna(subset=['mothers_epds_group_6-8weeks'])
df_epds_6to8weeks['mothers_epds_group_6-8weeks'].value_counts(dropna=False)
# TableOne for mothers_epds_group_6-8weeks
mytable_epds_6to8weeks = TableOne(df_epds_6to8weeks, columns, categorical, groupby = "mothers_epds_group_6-8weeks",rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_epds_6to8weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)


/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not found: {k}")


Grouped by mothers_epds_group_6-8weeks                                            
                                                                                                       Missing     Overall    Negative    Positive P-Value
n                                                                                                                      549         516          33        
Mother's Age, mean (SD)                                                                                      0  27.0 (6.4)  27.0 (6.5)  27.5 (5.8)   0.660
Mother's Age Group, n (%)          15-19 years                                                                   71 (12.9)   67 (13.0)    4 (12.1)   0.486
                                   20-24 years                                                                  143 (26.0)  138 (26.7)    5 (15.2)        
                                   25-29 years                                                                  147 (26.8)  134 (26.0)   13 (39.4)        
                                   30-34 years                                                                  114 (20.8)  106 (20.5)    8 (24.2)        
                                   35-39 years                                                                    54 (9.8)   52 (10.1)     2 (6.1)        
                                   40+ years                                                                      20 (3.6)    19 (3.7)     1 (3.0)        
Mother's Marital Status, n (%)     Divorced                                                                        1 (0.2)     1 (0.2)     0 (0.0)   0.915
                                   In a relationship                                                            102 (18.6)   97 (18.8)    5 (15.2)        
                                   Married                                                                      426 (77.6)  399 (77.3)   27 (81.8)        
                                   Separated                                                                       9 (1.6)     8 (1.6)     1 (3.0)        
                                   Single                                                                         10 (1.8)    10 (1.9)     0 (0.0)        
                                   Widowed                                                                         1 (0.2)     1 (0.2)     0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                               72 (13.1)   68 (13.2)    4 (12.1)   0.898
                                   Completed primary school education                                            79 (14.4)   73 (14.1)    6 (18.2)        
                                   Completed secondary school education                                          93 (16.9)   90 (17.4)     3 (9.1)        
                                   No education                                                                   14 (2.6)    13 (2.5)     1 (3.0)        
                                   Some college or university                                                     27 (4.9)    26 (5.0)     1 (3.0)        
                                   Some primary school education                                                  30 (5.5)    27 (5.2)     3 (9.1)        
                                   Some secondary school education                                              218 (39.7)  204 (39.5)   14 (42.4)        
                                   Vocational Training                                                            16 (2.9)    15 (2.9)     1 (3.0)        
Mother's Source of Income , n (%)  Formal Employment                                                             87 (15.8)   85 (16.5)     2 (6.1)   0.246
                                   Informal Employment                                                          145 (26.4)  134 (26.0)   11 (33.3)        
                                   Unemplo

In [397]:
df['fp_m_epds_harming_self'] = df['fp_m_epds_harming_self'].replace("None", np.nan,regex=True)
df_selfharm_2weeks = df.dropna(subset=['fp_m_epds_harming_self'])
df_selfharm_2weeks['fp_m_epds_harming_self'].value_counts(dropna=False)
# TableOne for fp_m_epds_harming_self
mytable_selfharm_2weeks = TableOne(df_selfharm_2weeks, columns, categorical, groupby = "fp_m_epds_harming_self",rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_selfharm_2weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not

Grouped by fp_m_epds_harming_self                                                                      
                                                                                                  Missing     Overall    Negative Positive(hardly-ever/Never/Sometimes) P-Value
n                                                                                                                 576         560                                    16        
Mother's Age, mean (SD)                                                                                 0  26.9 (6.5)  27.0 (6.5)                            24.7 (5.2)   0.103
Mother's Age Group, n (%)          15-19 years                                                              77 (13.4)   74 (13.2)                              3 (18.8)   0.421
                                   20-24 years                                                             152 (26.4)  148 (26.4)                              4 (25.0)        
                                   25-29 years                                                             152 (26.4)  145 (25.9)                              7 (43.8)        
                                   30-34 years                                                             118 (20.5)  116 (20.7)                              2 (12.5)        
                                   35-39 years                                                               54 (9.4)    54 (9.6)                               0 (0.0)        
                                   40+ years                                                                 23 (4.0)    23 (4.1)                               0 (0.0)        
Mother's Marital Status, n (%)     Divorced                                                                   1 (0.2)     1 (0.2)                               0 (0.0)   0.027
                                   In a relationship                                                       116 (20.1)  112 (20.0)                              4 (25.0)        
                                   Married                                                                 437 (75.9)  428 (76.4)                              9 (56.2)        
                                   Separated                                                                 10 (1.7)     9 (1.6)                               1 (6.2)        
                                   Single                                                                    11 (1.9)     9 (1.6)                              2 (12.5)        
                                   Widowed                                                                    1 (0.2)     1 (0.2)                               0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                          72 (12.5)   70 (12.5)                              2 (12.5)   0.382
                                   Completed primary school education                                       83 (14.4)   80 (14.3)                              3 (18.8)        
                                   Completed secondary school education                                     96 (16.7)   90 (16.1)                              6 (37.5)        
                                   No education                                                              14 (2.4)    14 (2.5)                               0 (0.0)        
                                   Some college or university                                                25 (4.3)    24 (4.3)                               1 (6.2)        
                                   Some primary school education                                             32 (5.6)    32 (5.7)                               0 (0.0)        
                                   Some secondary school education                                         238 (41.3)  234 (41.8)                              4 (25.0)        
                        

In [398]:
df['fp1_m_epds_harming_self'] = df['fp1_m_epds_harming_self'].replace("None", np.nan,regex=True)
df_selfharm_6to8weeks = df.dropna(subset=['fp1_m_epds_harming_self'])
# TableOne for fp1_m_epds_harming_self
mytable_selfharm_6to8weeks = TableOne(df_selfharm_6to8weeks, columns, categorical, groupby = "fp1_m_epds_harming_self",rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_selfharm_6to8weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not

Grouped by fp1_m_epds_harming_self                                                                      
                                                                                                   Missing     Overall    Negative Positive(hardly-ever/Never/Sometimes) P-Value
n                                                                                                                  542         530                                    12        
Mother's Age, mean (SD)                                                                                  0  27.0 (6.4)  27.0 (6.4)                            26.3 (6.1)   0.701
Mother's Age Group, n (%)          15-19 years                                                               70 (12.9)   67 (12.6)                              3 (25.0)   0.790
                                   20-24 years                                                              141 (26.0)  139 (26.2)                              2 (16.7)        
                                   25-29 years                                                              146 (26.9)  143 (27.0)                              3 (25.0)        
                                   30-34 years                                                              112 (20.7)  109 (20.6)                              3 (25.0)        
                                   35-39 years                                                                53 (9.8)    52 (9.8)                               1 (8.3)        
                                   40+ years                                                                  20 (3.7)    20 (3.8)                               0 (0.0)        
Mother's Marital Status, n (%)     Divorced                                                                    1 (0.2)     1 (0.2)                               0 (0.0)   0.627
                                   In a relationship                                                        102 (18.8)   99 (18.7)                              3 (25.0)        
                                   Married                                                                  419 (77.3)  411 (77.5)                              8 (66.7)        
                                   Separated                                                                   9 (1.7)     9 (1.7)                               0 (0.0)        
                                   Single                                                                     10 (1.8)     9 (1.7)                               1 (8.3)        
                                   Widowed                                                                     1 (0.2)     1 (0.2)                               0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                           72 (13.3)   70 (13.2)                              2 (16.7)   0.968
                                   Completed primary school education                                        78 (14.4)   76 (14.3)                              2 (16.7)        
                                   Completed secondary school education                                      92 (17.0)   91 (17.2)                               1 (8.3)        
                                   No education                                                               14 (2.6)    14 (2.6)                               0 (0.0)        
                                   Some college or university                                                 27 (5.0)    26 (4.9)                               1 (8.3)        
                                   Some primary school education                                              29 (5.4)    28 (5.3)                               1 (8.3)        
                                   Some secondary school education                                          215 (39.7)  210 (39.6)                              5 (41.7)        
 

In [399]:
df['anxiety_result_2weeks'] = df['anxiety_result_2weeks'].replace("None", np.nan,regex=True)
df_anxiety_2weeks = df.dropna(subset=['anxiety_result_2weeks'])
# TableOne for anxiety_result_2weeks
mytable_anxiety_2weeks = TableOne(df_anxiety_2weeks, columns, categorical, groupby = "anxiety_result_2weeks",rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_anxiety_2weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not

Grouped by anxiety_result_2weeks                                            
                                                                                                 Missing     Overall    Negative    Positive P-Value
n                                                                                                                576         446         130        
Mother's Age, mean (SD)                                                                                0  26.9 (6.5)  26.9 (6.4)  26.9 (6.6)   0.995
Mother's Age Group, n (%)          15-19 years                                                             77 (13.4)   58 (13.0)   19 (14.6)   0.459
                                   20-24 years                                                            152 (26.4)  121 (27.1)   31 (23.8)        
                                   25-29 years                                                            152 (26.4)  116 (26.0)   36 (27.7)        
                                   30-34 years                                                            118 (20.5)   87 (19.5)   31 (23.8)        
                                   35-39 years                                                              54 (9.4)   47 (10.5)     7 (5.4)        
                                   40+ years                                                                23 (4.0)    17 (3.8)     6 (4.6)        
Mother's Marital Status, n (%)     Divorced                                                                  1 (0.2)     1 (0.2)     0 (0.0)   0.074
                                   In a relationship                                                      116 (20.1)   91 (20.4)   25 (19.2)        
                                   Married                                                                437 (75.9)  342 (76.7)   95 (73.1)        
                                   Separated                                                                10 (1.7)     4 (0.9)     6 (4.6)        
                                   Single                                                                   11 (1.9)     7 (1.6)     4 (3.1)        
                                   Widowed                                                                   1 (0.2)     1 (0.2)     0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                         72 (12.5)   52 (11.7)   20 (15.4)   0.331
                                   Completed primary school education                                      83 (14.4)   59 (13.2)   24 (18.5)        
                                   Completed secondary school education                                    96 (16.7)   79 (17.7)   17 (13.1)        
                                   No education                                                             14 (2.4)    11 (2.5)     3 (2.3)        
                                   Some college or university                                               25 (4.3)    18 (4.0)     7 (5.4)        
                                   Some primary school education                                            32 (5.6)    22 (4.9)    10 (7.7)        
                                   Some secondary school education                                        238 (41.3)  193 (43.3)   45 (34.6)        
                                   Vocational Training                                                      16 (2.8)    12 (2.7)     4 (3.1)        
Mother's Source of Income , n (%)  Formal Employment                                                       87 (15.1)   67 (15.0)   20 (15.4)   0.308
                                   Informal Employment                                                    149 (25.9)  122 (27.4)   27 (20.8)        
                                   Unemployed                                                             340 (59.0)  257 (57.6)   83 (63.8)        
Mother's Alcohol Use Status, n (%) No            

In [400]:
df['anxiety_result_6to8weeks'] = df['anxiety_result_6to8weeks'].replace("None", np.nan,regex=True)
df_anxiety_6to8weeks = df.dropna(subset=['anxiety_result_6to8weeks'])
# TableOne for anxiety_result_6to8weeks
mytable_anxiety_6to8weeks = TableOne(df_anxiety_6to8weeks, columns, categorical, groupby = "anxiety_result_6to8weeks",rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)
mytable_anxiety_6to8weeks

/opt/anaconda3/lib/python3.12/site-packages/tableone/tableone.py:325: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self._categorical] = handle_categorical_nulls(data[self._categorical], self._categorical)
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_par_group
  warnings.warn(f"Order variable not found: {k}")
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:184: UserWarning: Order value not found: "mothers_grav_group: ['0']"
  warnings.warn(f'Order value not found: "{k}: {rm_var}"')
/opt/anaconda3/lib/python3.12/site-packages/tableone/formatting.py:177: UserWarning: Order variable not found: mothers_sex_debut_group
  warnings.warn(f"Order variable not

Grouped by anxiety_result_6to8weeks                                            
                                                                                                    Missing     Overall    Negative    Positive P-Value
n                                                                                                                   542         480          62        
Mother's Age, mean (SD)                                                                                   0  27.0 (6.4)  27.0 (6.5)  26.8 (6.2)   0.809
Mother's Age Group, n (%)          15-19 years                                                                70 (12.9)   63 (13.1)    7 (11.3)   0.929
                                   20-24 years                                                               141 (26.0)  123 (25.6)   18 (29.0)        
                                   25-29 years                                                               146 (26.9)  129 (26.9)   17 (27.4)        
                                   30-34 years                                                               112 (20.7)   98 (20.4)   14 (22.6)        
                                   35-39 years                                                                 53 (9.8)   49 (10.2)     4 (6.5)        
                                   40+ years                                                                   20 (3.7)    18 (3.8)     2 (3.2)        
Mother's Marital Status, n (%)     Divorced                                                                     1 (0.2)     1 (0.2)     0 (0.0)   0.903
                                   In a relationship                                                         102 (18.8)   90 (18.8)   12 (19.4)        
                                   Married                                                                   419 (77.3)  370 (77.1)   49 (79.0)        
                                   Separated                                                                    9 (1.7)     8 (1.7)     1 (1.6)        
                                   Single                                                                      10 (1.8)    10 (2.1)     0 (0.0)        
                                   Widowed                                                                      1 (0.2)     1 (0.2)     0 (0.0)        
Mother's Education Status, n (%)   Completed college or university                                            72 (13.3)   66 (13.8)     6 (9.7)   0.140
                                   Completed primary school education                                         78 (14.4)   69 (14.4)    9 (14.5)        
                                   Completed secondary school education                                       92 (17.0)   86 (17.9)     6 (9.7)        
                                   No education                                                                14 (2.6)    11 (2.3)     3 (4.8)        
                                   Some college or university                                                  27 (5.0)    21 (4.4)     6 (9.7)        
                                   Some primary school education                                               29 (5.4)    25 (5.2)     4 (6.5)        
                                   Some secondary school education                                           215 (39.7)  191 (39.8)   24 (38.7)        
                                   Vocational Training                                                         15 (2.8)    11 (2.3)     4 (6.5)        
Mother's Source of Income , n (%)  Formal Employment                                                          86 (15.9)   79 (16.5)    7 (11.3)   0.153
                                   Informal Employment                                                       142 (26.2)  130 (27.1)   12 (19.4)        
                                   Unemployed                                                                314 (57.9) 

In [401]:
# Create an Excel writer object
with pd.ExcelWriter('/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/Analysis/exploratory/Maternal_EPDS_Anxiety analysis_tables.xlsx', engine='openpyxl') as writer:
    
    mytable.tableone.to_excel(writer, sheet_name='Martenal Demographics' )
    mytable_epds_2weeks.tableone.to_excel(writer, sheet_name='EPDS 2 weeks')
    mytable_epds_6to8weeks.tableone.to_excel(writer, sheet_name='EPDS 6-8 weeks')
    mytable_selfharm_2weeks.tableone.to_excel(writer, sheet_name='Self-Harm 2 weeks')
    mytable_selfharm_6to8weeks.tableone.to_excel(writer, sheet_name='Self-Harm 6-8 weeks')
    mytable_anxiety_2weeks.tableone.to_excel(writer, sheet_name='Anxiety 2 weeks')
    mytable_anxiety_6to8weeks.tableone.to_excel(writer, sheet_name='Anxiety 6-8 weeks')

# logistic GEE analysis

In [402]:
# Drop missing epds scores at 6-8 weeks
df = df.dropna(subset=['fp1_m_scr_epds_sco'])


df['2weeks'] = df['mothers_epds_group_2weeks']
df['6_8weeks'] = df['mothers_epds_group_6-8weeks']

# Copy database for logistic regression analysis
df_logit = df[['participant_id','2weeks','6_8weeks','en_rand']].copy()

# data variable transformation for logistic regression
# Randomisation group
df_logit['en_rand'] = df_logit['en_rand'].map({'Control (NICU → KMC ward for < 48 hours → home)': 'Control', 'Intervention (Immediate admission to the KMC Ward → home)': 'Intervention'})
df_logit['en_rand_num'] = df_logit['en_rand'].map({ 'Control' : 0, 'Intervention' : 1 })
# Timepoint



/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/2684455531.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['2weeks'] = df['mothers_epds_group_2weeks']
/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_75077/2684455531.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['6_8weeks'] = df['mothers_epds_group_6-8weeks']


In [403]:
# tranform dataframe from wide to long format for logistic regression analysis

df_logit_long = pd.melt(
    df_logit, 
    id_vars=['participant_id','en_rand'], 
    value_vars=['2weeks', '6_8weeks'], 
    var_name='timepoint', value_name='epds_result')

df_logit_long.sort_values(by=['participant_id'], inplace=True)

# Change the 'timepoint' and 'epds_result' columns to numeric values for logistic regression
df_logit_long['timepoint_num'] = df_logit_long['timepoint'].map({'2weeks': 0, '6_8weeks': 1})
df_logit_long['epds_result_num'] = df_logit_long['epds_result'].map({'Negative': 0, 'Positive': 1})


In [404]:
# check for duplicate ids
duplicates = df_logit_long[
    df_logit_long.duplicated(
        subset=["participant_id", "timepoint"],
        keep=False
    )
]

print(duplicates)

Empty DataFrame
Columns: [participant_id, en_rand, timepoint, epds_result, timepoint_num, epds_result_num]
Index: []


In [405]:
#checking for available epds data
participant_counts = (
    df_logit_long
    .groupby("participant_id")["epds_result"]
    .nunique()
)

print(participant_counts.value_counts())

epds_result
1    460
2     89
Name: count, dtype: int64


In [406]:
df_logit_long['epds_result'].value_counts(dropna=False)

epds_result
Negative    978
Positive    115
NaN           5
Name: count, dtype: int64

In [407]:
summary = (
    df_logit_long
    .groupby(["en_rand", "timepoint"])
    .agg(
        n=("epds_result_num", "size"),
        positive=("epds_result_num", "sum"),
        positivity=("epds_result_num", "mean")
    )
    .reset_index()
)

summary["positivity_percent"] = summary["positivity"] * 100

print(summary)

Empty DataFrame
Columns: [en_rand, timepoint, n, positive, positivity, positivity_percent]
Index: []


In [408]:
positivity_table = df_logit_long.pivot_table(
    index="en_rand",
    columns="timepoint",
    values="epds_result_num",
    aggfunc="mean"
) * 100

print(positivity_table)

Empty DataFrame
Columns: []
Index: []


In [409]:
positivity_table

timepoint
en_rand


In [410]:
positivity_table["change"] = (
    positivity_table['2weeks']
    - positivity_table['6_8weeks']
)

print(positivity_table)

KeyError: '2weeks'

# Logistic GEE

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
df_logit_long["group"] = pd.Categorical(
    df_logit_long["en_rand"],
    categories=["Control", "Intervention"]
)

df_logit_long["timepoint"] = pd.Categorical(
    df_logit_long["timepoint"],
    categories=["2weeks", "6_8weeks"]
)

In [ ]:
model = smf.gee(
    formula="epds_result_num ~ group * timepoint",
    groups="participant_id",
    data=df_logit_long,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result = model.fit()

print(result.summary())

In [ ]:
params = result.params
conf = result.conf_int()

or_table = pd.DataFrame({
    "OR": np.exp(params),
    "CI_lower": np.exp(conf[0]),
    "CI_upper": np.exp(conf[1]),
    "p_value": result.pvalues
})

print(or_table)

In [ ]:
# Get predicted positivity probabilities

prediction_df = pd.DataFrame({
    "group": [
        "Control",
        "Control",
        "Intervention",
        "Intervention"
    ],
    "timepoint": [
        "2weeks",
        "6_8weeks",
        "2weeks",
        "6_8weeks"
    ]
})

prediction_df["predicted_probability"] = (
    result.predict(prediction_df)
)

prediction_df["predicted_percent"] = (
    prediction_df["predicted_probability"] * 100
)

print(prediction_df)


In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    df_logit_long
    .groupby(["group", "timepoint"], observed=False)["epds_result_num"]
    .mean()
    .reset_index()
)

plot_df["positivity_percent"] = (
    plot_df["epds_result_num"] * 100
)

for group_name, temp in plot_df.groupby("group"):
    plt.plot(
        temp["timepoint"],
        temp["positivity_percent"],
        marker="o",
        label=group_name.capitalize()
    )

plt.xlabel("Study time")
plt.ylabel("Positivity (%)")
plt.title("Change in Positivity from Baseline to Follow-up")
plt.legend(title="Study group")
plt.tight_layout()
plt.show()

#### Interpretation of GEE Regression Results

A generalized estimating equation (GEE) model with a binomial distribution and exchangeable correlation structure was used to assess changes in the odds of a positive EPDS result between **2 weeks** and **6–8 weeks postpartum**, and to determine whether this change differed between the **control** and **intervention** groups.

The analysis included **1,131 observations from 590 participants**, with a mean of 1.9 observations per participant.

#### Main Findings

At **2 weeks postpartum**, participants in the intervention group had significantly lower odds of a positive EPDS result compared with participants in the control group (**OR = 0.61, 95% CI: 0.38–0.98, p = 0.041**). This corresponds to approximately **39% lower odds of EPDS positivity** in the intervention group at the baseline timepoint.

Among participants in the **control group**, the odds of a positive EPDS result were significantly lower at **6–8 weeks** compared with **2 weeks postpartum** (**OR = 0.41, 95% CI: 0.25–0.68, p < 0.001**). This corresponds to approximately **59% lower odds of positivity** at follow-up.

However, the **group × time interaction** was not statistically significant (**OR = 0.78, 95% CI: 0.34–1.77, p = 0.547**). This indicates that there was **no statistically significant evidence that the change in EPDS positivity from 2 weeks to 6–8 weeks differed between the intervention and control groups**.

#### GEE Model Results

| Model term | Coefficient (β) | Adjusted OR | 95% CI for OR | p-value | Interpretation |
|---|---:|---:|---:|---:|---|
| Intercept | -1.563 | 0.21 | 0.15–0.28 | <0.001 | Odds of positivity in the control group at 2 weeks |
| Intervention vs Control at 2 weeks | -0.491 | 0.61 | 0.38–0.98 | **0.041** | Intervention group had lower baseline odds of positivity |
| 6–8 weeks vs 2 weeks in Control | -0.882 | 0.41 | 0.25–0.68 | **<0.001** | Positivity odds decreased significantly over time in the control group |
| Intervention × 6–8 weeks | -0.253 | 0.78 | 0.34–1.77 | 0.547 | No statistically significant difference in change over time between groups |

**Reference categories:** Control group and 2-week timepoint.

#### Interpretation of the Intervention Effect

The most important estimate for assessing the effect of the intervention is the **group × time interaction**:

\[
OR_{\text{interaction}} = 0.78
\]

Although the point estimate is below 1, suggesting that the intervention group may have experienced a somewhat greater reduction in positivity over time than the control group, the 95% confidence interval is wide and includes 1:

\[
95\%\,CI = 0.34 - 1.77
\]

The interaction was also not statistically significant:

\[
p = 0.547
\]

Therefore, the study does **not provide evidence that the intervention produced a significantly greater reduction in EPDS positivity between 2 weeks and 6–8 weeks compared with the control group**.

An important feature of these results is that the intervention and control groups were already significantly different at 2 weeks. The intervention group had lower odds of positivity at the initial measurement. This means that simply comparing the two groups at follow-up could be misleading. The interaction term is therefore the appropriate estimate for evaluating whether the **change over time differed between groups**.

#### Suggested Scientific Reporting

> A binomial generalized estimating equation model with an exchangeable working correlation structure was used to examine differences in EPDS positivity between the intervention and control groups over time. The analysis included 1,131 observations from 590 participants. At 2 weeks postpartum, participants in the intervention group had significantly lower odds of EPDS positivity compared with those in the control group (OR = 0.61, 95% CI: 0.38–0.98; p = 0.041). In the control group, the odds of EPDS positivity decreased significantly between 2 weeks and 6–8 weeks postpartum (OR = 0.41, 95% CI: 0.25–0.68; p < 0.001). However, there was no significant group-by-time interaction (OR = 0.78, 95% CI: 0.34–1.77; p = 0.547), indicating that the reduction in EPDS positivity over time did not differ significantly between the intervention and control groups.

#### Overall Conclusion

The results suggest that EPDS positivity decreased over time, particularly within the control group. Although the intervention group had lower odds of positivity at 2 weeks, there was **no statistically significant evidence that the intervention group experienced a greater reduction in positivity over time compared with the control group**.

The absence of a statistically significant interaction should not be interpreted as proof that the intervention had no effect. Rather, the findings indicate that the available data did not provide sufficient evidence of a differential change between the two study groups during the observed follow-up period.

# MSSRP Abstract Due 28-08-2026

In [ ]:
df_mssrp

In [ ]:
df_mssrp['en_m_hiv'].value_counts()

In [ ]:
df_mssrp['en_hiv_exposure'] = df_mssrp['en_m_hiv'].map({'Positive': 'HIV-exposed', 'Negative': 'HIV-unexposed',"Unknown": "HIV-unknown"})

In [ ]:
columns_t1 = [ 
    #Neonatal variables
    "en_bw",
    "en_gaw",
    "en_hgt",
    "en_wgt",
    "en_hiv_exposure",
    #Maternal variables
    "mothers_age_group",
    "en_m_hgt",
    "en_m_wgt",
    "mothers_par_group",
    "mothers_grav_group",
    "en_m_mstat",
    "en_m_hiv",
]
categorical_t1 = [
#Neonatal variables
    "en_hiv_exposure",
    #Maternal variables
    "mothers_age_group",
    "mothers_par_group",
    "mothers_grav_group",
    "en_m_mstat",
    "en_m_hiv",
]

continuous_t1 = [
    "en_m_hgt",
    "en_m_wgt",
    "en_hgt",
    "en_wgt",
    "en_bw",]
min_max_t1 = [
    "en_bw",
    "en_gaw",
    "en_m_hgt",
    "en_m_wgt",
    "en_hgt",
    "en_wgt"]
nonnormal_t1 = ["mothers_par_group","mothers_grav_group"] 



lables = {
    "en_bw": "Neonatal Birth Weight",
    "en_gaw": "Neonatal Gestational Age",
    "en_hgt": "Neonatal Height",
    "en_wgt": "Neonatal Weight",
    "en_hiv_exposure": "Neonatal HIV Exposure",
    "mothers_age_group": "Mother's Age Group",
    "mothers_par_group": "Mother's Parity Group",
    "mothers_grav_group": "Mother's Gravida Group",
    "en_misc": "Miscarriages",
    "en_mod": "Mode of Delivery",
    "en_m_hgt": "Mother's Height",
    "en_m_wgt": "Mother's Weight",
    "mothers_sex_debut_group": "Mother's Sexual Debut Age Group",
    "en_m_mstat": "Mother's Marital Status",
    "en_m_hiv": "Mother's HIV Status",
    "en_m_sku": "Mother's Education Status",
    "en_m_inc": "Mother's Source of Income ",
    "en_m_mlev": "Maternal leave",
    "mothers_anc_group": "Mother's ANC Visits",
    "en_m_smk": "Mother's Smoking Status",
    "en_m_drg": "Mother's Drug Use Status",
    "en_m_alc": "Mother's Alcohol Use Status"
}   

groupby = 'en_rand'

In [ ]:
df_mssrp['en_m_hiv'] = df_mssrp['en_m_hiv'].replace('None',"Unknown",regex=True)
df_mssrp['en_m_hiv'].value_counts()

In [ ]:
MSSRP_Abstract = TableOne(df_mssrp, columns_t1, categorical_t1, continuous_t1, groupby = "en_rand",min_max=min_max_t1,rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)

In [ ]:
MSSRP_Abstract.head(100)

# ASQ Tables 

In [ ]:
#df.columns.values[:1000]

In [ ]:
# asq data agmentation
# change columns to categories for analysis
df_mssrp['Communication(Black_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_i'],
                                  bins=[-1,29.65, 1000],
                                  labels=[ 'Positive','Negative'])    
df_mssrp['Gross Motor(Black_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_ii'],
                                  bins=[-1,22.25, 1000],
                                  labels=[ 'Positive','Negative'])
df_mssrp['Fine Motor(Black_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_iii'],
                                  bins=[-1,25.14, 1000],
                                  labels=[ 'Positive', 'Negative'])
df_mssrp['Problem Solving(Black_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_iv'],
                                  bins=[-1,27.72, 1000],
                                  labels=['Positive','Negative'])
df_mssrp['Personal Social(Black_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_v'],
                                  bins=[-1,25.34, 1000],        
                                  labels=[ 'Positive', 'Negative'])
# Grey cutoff categorization
df_mssrp['Communication(Grey_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_i'],
                                  bins=[-1,39.27, 1000],
                                  labels=[ 'Positive','Negative'])    
df_mssrp['Gross Motor(Grey_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_ii'],
                                  bins=[-1,33.95, 1000],
                                  labels=[ 'Positive','Negative'])
df_mssrp['Fine Motor(Grey_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_iii'],
                                  bins=[-1,37.04, 1000],
                                  labels=[ 'Positive', 'Negative'])
df_mssrp['Problem Solving(Grey_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_iv'],
                                  bins=[-1,39.06, 1000],
                                  labels=['Positive','Negative'])
df_mssrp['Personal Social(Grey_cutoff)'] = pd.cut(df_mssrp['fp2_m_inf_asq_y_v'],
                                  bins=[-1,36.83, 1000],        
                                  labels=[ 'Positive', 'Negative'])
df_mssrp[["fp2_m_inf_asq_y_i",
"fp2_m_inf_asq_y_ii",
"fp2_m_inf_asq_y_iii",
"fp2_m_inf_asq_y_iv",
"fp2_m_inf_asq_y_v",
#black cutoff categorization
"Communication(Black_cutoff)",
"Gross Motor(Black_cutoff)",
"Fine Motor(Black_cutoff)",
"Problem Solving(Black_cutoff)", 
"Personal Social(Black_cutoff)",
#grey cutoff categorization
"Communication(Grey_cutoff)",
"Gross Motor(Grey_cutoff)",
"Fine Motor(Grey_cutoff)",
"Problem Solving(Grey_cutoff)", 
"Personal Social(Grey_cutoff)"]]

In [ ]:
columns_asq = [ 
    #black cutoff categorization
    "Communication(Black_cutoff)",
    "Gross Motor(Black_cutoff)",
    "Fine Motor(Black_cutoff)",
    "Problem Solving(Black_cutoff)", 
    "Personal Social(Black_cutoff)",
    #grey cutoff categorization
    "Communication(Grey_cutoff)",
    "Gross Motor(Grey_cutoff)",
    "Fine Motor(Grey_cutoff)",
    "Problem Solving(Grey_cutoff)", 
    "Personal Social(Grey_cutoff)"
]
categorical_asq = [
    #black cutoff categorization
    "Communication(Black_cutoff)",
    "Gross Motor(Black_cutoff)",
    "Fine Motor(Black_cutoff)",
    "Problem Solving(Black_cutoff)", 
    "Personal Social(Black_cutoff)",
    #grey cutoff categorization
    "Communication(Grey_cutoff)",
    "Gross Motor(Grey_cutoff)",
    "Fine Motor(Grey_cutoff)",
    "Problem Solving(Grey_cutoff)", 
    "Personal Social(Grey_cutoff)"
]

groupby_asq = 'en_rand'

In [ ]:
df_mssrp[columns_asq+['mothers_epds_group_2weeks']] = df_mssrp[columns_asq+['mothers_epds_group_2weeks']].replace("None", np.nan)
df_mssrp = df_mssrp.dropna(
    subset=columns_asq,
    how="all"
)
df_mssrp

In [ ]:
MSSRP_Abstract_asq = TableOne(df_mssrp, columns_asq, categorical_asq, groupby = groupby_asq,  pval=True)
MSSRP_Abstract_asqbyEpds = TableOne(df_mssrp, columns_asq, categorical_asq, groupby = "mothers_epds_group_2weeks",  pval=True)

In [ ]:
MSSRP_Abstract_asq

In [ ]:
MSSRP_Abstract_asqbyEpds

In [ ]:
# Create an Excel writer object
with pd.ExcelWriter('/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/Analysis/MSSRP/MSSRP_ppd_tables.xlsx', engine='openpyxl') as writer:
    # Write the first table to the first sheet
    MSSRP_Abstract.tableone.to_excel(writer, sheet_name='Martenal & Neonatal')
    
    # Write the second table to the second sheet
    MSSRP_Abstract_asq.tableone.to_excel(writer, sheet_name='ASQ by cohort Table')
    # Write the Third table to the second sheet
    MSSRP_Abstract_asqbyEpds.tableone.to_excel(writer, sheet_name='ASQ by epds Table')

In [ ]:
df_rand = df.loc[(df['fp_m_scr_epds_sco'] >= 11) & (df["en_rand"] == 'Intervention (Immediate admission to the KMC Ward → home)')& (df["participant_id_f"].notna())]

In [ ]:
random_ids = (
    dedup_df["participant_id_f"]
    .dropna()
    .drop_duplicates()
    .sample(n=5)
    .tolist()
)

random_ids

In [ ]:
df_rand.shape

In [ ]:
df_rand['participant_id_f'].notna().value_counts()

In [ ]:
"025"
"102"